#Librerias

In [3]:
# ============================================================
# 0) IMPORTS BÁSICOS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Para ver más filas si quieres
pd.set_option("display.max_rows", 200)


#Modelos difusos

In [1]:
# ============================================================
# 1) PLANTILLAS DE CLASES DIFUSAS
# ============================================================
def crear_clase_fuzzy_Low(nombre_clase, offset, **conjuntos):
    """
    Fuzzy tipo 'Low': se usa un límite inferior (lmin) y se trabaja con offset = pv - lmin.
    """

    class FuzzyTemplate:
        def __init__(self):
            self.offset = np.array(offset, dtype=float)

            # Guardar los conjuntos difusos
            for nombre, valores in conjuntos.items():
                setattr(self, nombre, np.array(valores, dtype=float))

            # Crear funciones de pertenencia dinámicamente
            for nombre in conjuntos.keys():
                vector = getattr(self, nombre)
                setattr(
                    self,
                    f"mf_{nombre}",
                    lambda x, v=vector: np.interp(
                        x, self.offset, v, left=v[0], right=v[-1]
                    )
                )

        def calcular_offset(self, pv, lmin):
            return float(pv) - float(lmin)

        def evaluar(self, pv, lmin):
            offset = self.calcular_offset(pv, lmin)

            pertenencias = {}
            for nombre in conjuntos.keys():
                mf = getattr(self, f"mf_{nombre}")
                pertenencias[nombre] = float(np.clip(mf(offset), 0, 1))

            dominante = max(pertenencias, key=pertenencias.get)
            valor_dominante = pertenencias[dominante]
            return dominante, valor_dominante, offset, pertenencias

    FuzzyTemplate.__name__ = nombre_clase
    return FuzzyTemplate


def crear_clase_fuzzy_high(nombre_clase, offset, **conjuntos):
    """
    Fuzzy tipo 'High': se usa un límite superior (lmax) y offset = lmax - pv.
    """

    class FuzzyTemplate:
        def __init__(self):
            self.offset = np.array(offset, dtype=float)

            # Guardar los conjuntos difusos
            for nombre, valores in conjuntos.items():
                setattr(self, nombre, np.array(valores, dtype=float))

            # Crear funciones de pertenencia dinámicamente
            for nombre in conjuntos.keys():
                vector = getattr(self, nombre)
                setattr(
                    self,
                    f"mf_{nombre}",
                    lambda x, v=vector: np.interp(
                        x, self.offset, v, left=v[0], right=v[-1]
                    )
                )

        def calcular_offset(self, pv, lmax):
            return float(lmax) - float(pv)

        def evaluar(self, pv, lmax):
            offset = self.calcular_offset(pv, lmax)

            pertenencias = {}
            for nombre in conjuntos.keys():
                mf = getattr(self, f"mf_{nombre}")
                pertenencias[nombre] = float(np.clip(mf(offset), 0, 1))

            dominante = max(pertenencias, key=pertenencias.get)
            valor_dominante = pertenencias[dominante]
            return dominante, valor_dominante, offset, pertenencias

    FuzzyTemplate.__name__ = nombre_clase
    return FuzzyTemplate


def crear_clase_fuzzy_norm(nombre_clase, offset, **conjuntos):
    """
    Fuzzy normalizado: offset = (pv - lmin) / (lmax - lmin).
    """

    class FuzzyTemplate:
        def __init__(self):
            self.offset = np.array(offset, dtype=float)

            # Guardar los conjuntos difusos
            for nombre, valores in conjuntos.items():
                setattr(self, nombre, np.array(valores, dtype=float))

            # Crear funciones de pertenencia dinámicamente
            for nombre in conjuntos.keys():
                vector = getattr(self, nombre)
                setattr(
                    self,
                    f"mf_{nombre}",
                    lambda x, v=vector: np.interp(
                        x, self.offset, v, left=v[0], right=v[-1]
                    )
                )

        def calcular_offset(self, pv, lmin, lmax):
            return (float(pv) - float(lmin)) / (float(lmax) - float(lmin))

        def evaluar(self, pv, lmin, lmax):
            offset = self.calcular_offset(pv, lmin, lmax)

            pertenencias = {}
            for nombre in conjuntos.keys():
                mf = getattr(self, f"mf_{nombre}")
                pertenencias[nombre] = float(np.clip(mf(offset), 0, 1))

            dominante = max(pertenencias, key=pertenencias.get)
            valor_dominante = pertenencias[dominante]
            return dominante, valor_dominante, offset, pertenencias

    FuzzyTemplate.__name__ = nombre_clase
    return FuzzyTemplate


#Modelos difusos para crear variables

In [4]:
# ============================================================
# 2) MODELOS DIFUSOS PARA VARIABLES SAG
# ============================================================

# Potencia molino de bolas (no la usamos en las reglas de prueba, pero se evalúa igual)
PotMolBolas = crear_clase_fuzzy_Low(
    "pot_mol_bolas",
    offset=[-10, 0, 10, 20, 40, 60, 70, 80],
    bajo=[0, 0, 0, 0.5, 1, 1, 1, 1],
    OK=[0, 0.5, 1, 0.5, 0, 0, 0, 0],
    alto=[1, 0.5, 0, 0, 0, 0, 0, 0]
)
fuzzy_pot_bolas = PotMolBolas()

# Potencia SAG (límite superior)
PotSag1 = crear_clase_fuzzy_high(
    "pot_sag1",
    offset=[-10, 0, 10, 20, 40, 60, 70, 80],
    bajo=[0, 0, 0, 0.5, 1, 1, 1, 1],
    OK=[0, 0.5, 1, 0.5, 0, 0, 0, 0],
    alto=[1, 0.5, 0, 0, 0, 0, 0, 0]
)
fuzzy_pot_sag1 = PotSag1()

# Nivel (normalizado)
FuzzyNivel1 = crear_clase_fuzzy_norm(
    "fuzzy_nivel1",
    offset=[-1.1, -1, 0, 0.5, 0.7, 0.8, 1.0, 1.1],
    bajo=[0, 0, 0, 0.5, 1, 1, 1, 1],
    OK=[0, 0.5, 1, 0.5, 0, 0, 0, 0],
    alto=[1, 0.5, 0, 0, 0, 0, 0, 0]
)
fuzzy_nivel1 = FuzzyNivel1()

# P80 (límite superior)
P80Model = crear_clase_fuzzy_high(
    "p80_model",
    offset=[-20, 0, 20, 40, 60, 80, 100, 120],
    bajo=[1, 0.5, 0, 0, 0, 0, 0, 0],      # P80 muy bajo = sobre molienda
    OK=[0, 0.5, 1, 0.5, 0, 0, 0, 0],       # objetivo
    alto=[0, 0, 0, 0.5, 1, 1, 1, 1]        # P80 alto = problema
)
fuzzy_p80 = P80Model()

# Presión ciclones
PresionModel = crear_clase_fuzzy_high(
    "presion_model",
    offset=[-5, 0, 5, 10, 20, 30, 40, 50],
    bajo=[1, 0.5, 0, 0, 0, 0, 0, 0],
    OK=[0, 0.5, 1, 0.5, 0, 0, 0, 0],
    alto=[0, 0, 0, 0.5, 1, 1, 1, 1]
)
fuzzy_presion = PresionModel()

# Densidad
DensidadModel = crear_clase_fuzzy_high(
    "densidad_model",
    offset=[-10, 0, 10, 20, 30, 40, 50, 60],
    bajo=[1, 0.5, 0, 0, 0, 0, 0, 0],
    OK=[0, 0.5, 1, 0.5, 0, 0, 0, 0],
    alto=[0, 0, 0, 0.5, 1, 1, 1, 1]
)
fuzzy_densidad = DensidadModel()


In [5]:
# ============================================================
# 3) FUNCIÓN PARA EVALUAR TODAS LAS VARIABLES
# ============================================================
def evaluar_fuzzy(
    pv_bolas, lmin_bolas,
    pv_sag, lmax_sag,
    pv_nivel, lmin_nivel, lmax_nivel,
    pv_p80, lmax_p80,
    pv_pres, lmax_pres,
    pv_dens, lmax_dens,
):

    dom_bolas, val_bolas, off_bolas, pert_bolas = fuzzy_pot_bolas.evaluar(pv_bolas, lmin_bolas)
    dom_sag, val_sag, off_sag, pert_sag = fuzzy_pot_sag1.evaluar(pv_sag, lmax_sag)
    dom_nivel, val_nivel, off_nivel, pert_nivel = fuzzy_nivel1.evaluar(pv_nivel, lmin_nivel, lmax_nivel)

    dom_p80, val_p80, off_p80, pert_p80 = fuzzy_p80.evaluar(pv_p80, lmax_p80)
    dom_pres, val_pres, off_pres, pert_pres = fuzzy_presion.evaluar(pv_pres, lmax_pres)
    dom_dens, val_dens, off_dens, pert_dens = fuzzy_densidad.evaluar(pv_dens, lmax_dens)

    return {
        "pot_bolas": {
            "dom": dom_bolas, "val": val_bolas, "offset": off_bolas, "pert": pert_bolas},

        "pot_sag": {
            "dom": dom_sag, "val": val_sag, "offset": off_sag, "pert": pert_sag},

        "nivel": {
            "dom": dom_nivel, "val": val_nivel, "offset": off_nivel, "pert": pert_nivel},

        "p80": {
            "dom": dom_p80, "val": val_p80, "offset": off_p80, "pert": pert_p80},

        "presion": {
            "dom": dom_pres, "val": val_pres, "offset": off_pres, "pert": pert_pres},

        "densidad": {
            "dom": dom_dens, "val": val_dens, "offset": off_dens, "pert": pert_dens},
    }


#Evaluar condiciones

In [29]:
# ============================================================
# 4) EVALUAR CONDICIONES LINGÜÍSTICAS (ALTO, NO-ALTO, etc.)
# ============================================================
def evaluar_condicion(signal_fuzzy, condicion):
    """
    Evalúa condiciones tipo:
      - 'ALTO'
      - 'OK'
      - 'BAJO'
      - 'NO-ALTO'
      - 'NO-OK'
      - 'NO-BAJO'
    """
    cond = condicion.upper().strip()
    dom = str(signal_fuzzy["dom"]).upper()

    # Caso NO-XXXX
    if cond.startswith("NO-"):
        target = cond.replace("NO-", "")
        return dom != target

    # Caso normal
    return dom == cond



#Defuzzy

In [19]:
# ============================================================
# 5) ACCIONES FÍSICAS Y DEFUZZY
# ============================================================

ACCIONES_FISICAS = {
    "aumentar_tonelaje": 10,       # ton/h
    "disminuir_tonelaje": -10,     # ton/h
    "aumentar_agua_molino": 5,     # m3/h
    "disminuir_agua_molino": -5,   # m3/h
    "aumentar_agua_cajon": 4,
    "disminuir_agua_cajon": -4,
    "aumentar_agua_ciclones": 6,
    "disminuir_agua_ciclones": -6,
    "mantener": 0
}

def fuerza_de_regla(result, regla):
    fuerzas = []
    for var, cond in regla["si"]:
        fuerzas.append(result[var]["val"])
    return min(fuerzas)

def defuzzy(result, salida_motor):
    if salida_motor["regla_activada"] is None:
        return {"accion_fisica": 0, "comentario": "sin acción"}

    # localizar la regla dentro del bloque de prueba
    regla = None
    for r in REGLAS_TEST[salida_motor["bloque"]]:
        if r["id"] == salida_motor["regla_activada"]:
            regla = r
            break

    fuerza = fuerza_de_regla(result, regla)
    accion = salida_motor["accion"]
    max_step = ACCIONES_FISICAS[accion]
    step = fuerza * max_step

    return {
        "accion": accion,
        "fuerza": fuerza,
        "step": step,
        "unidad": "ton/h" if "tonelaje" in accion else "m3/h"
    }



#Reglas

In [27]:
# ============================================================
# 6) REGLAS DE PRUEBA (3–4 REGLAS) Y MOTOR
# ============================================================

# ==========================
# REGLAS DE PRUEBA FINALES
# ==========================
REGLAS_TEST = {
    "potencia_sag_test": [
        {
            "id": 1,
            "descripcion": "Potencia SAG ALTA y P80 ALTO → aumentar tonelaje",
            "si": [
                ("pot_sag", "ALTO"),
                ("p80", "ALTO"),
            ],
            "accion": "aumentar_tonelaje",
            "duracion": 60,
            "prioridad": 1
        },
        {
            "id": 2,
            "descripcion": "Potencia SAG OK y P80 OK → disminuir tonelaje",
            "si": [
                ("pot_sag", "OK"),
                ("p80", "OK"),
            ],
            "accion": "disminuir_tonelaje",
            "duracion": 60,
            "prioridad": 1
        },
    ],

    "control_p80_test": [
        {
            "id": 3,
            "descripcion": "P80 BAJO y presión OK → aumentar agua ciclones",
            "si": [
                ("p80", "BAJO"),
                ("presion", "OK"),
            ],
            "accion": "aumentar_agua_ciclones",
            "duracion": 60,
            "prioridad": 1
        },
        {
            "id": 4,
            "descripcion": "P80 ALTO y presión ALTA → disminuir agua ciclones",
            "si": [
                ("p80", "ALTO"),
                ("presion", "ALTO"),
            ],
            "accion": "disminuir_agua_ciclones",
            "duracion": 60,
            "prioridad": 1
        },
    ]
}


# Diccionario global de cooldown para las reglas de prueba
cooldown_until_test = {}   # {id_regla: tiempo_sim_hasta_que_expira}

def motor_reglas_test(result, reglas, tiempo_actual):
    """
    Motor de reglas de PRUEBA con tiempo simulado.
    """

    # 1) Ordenar bloques por prioridad
    bloques_ordenados = sorted(
        reglas.items(),
        key=lambda x: min(r["prioridad"] for r in x[1]) if x[1] else 999
    )

    # 2) Recorrer bloques y reglas
    for nombre_bloque, lista_reglas in bloques_ordenados:
        for regla in lista_reglas:
            rid = regla["id"]
            dur = regla["duracion"]

            # Cooldown
            t_cool = cooldown_until_test.get(rid, -1e12)
            if tiempo_actual < t_cool:
                continue

            # Evaluar condiciones
            cumple = True
            for var, cond_texto in regla["si"]:
                if not evaluar_condicion(result[var], cond_texto):
                    cumple = False
                    break

            if cumple:
                cooldown_until_test[rid] = tiempo_actual + dur
                return {
                    "regla_activada": rid,
                    "bloque": nombre_bloque,
                    "descripcion": regla["descripcion"],
                    "accion": regla["accion"],
                    "duracion": dur,
                    "cooldown_hasta": cooldown_until_test[rid],
                }

    return {
        "regla_activada": None,
        "bloque": None,
        "descripcion": "No se activó ninguna regla",
        "accion": "sin_accion",
        "duracion": 0,
        "cooldown_hasta": None,
    }


#Prueba

In [20]:
# ============================================================
# 7) CICLO DE CONTROL DE PRUEBA
# ============================================================
def tick_test(variables, tiempo_actual):
    result = evaluar_fuzzy(
        pv_bolas=variables["pot_bolas"], lmin_bolas=variables["lmin_bolas"],
        pv_sag=variables["pot_sag"],     lmax_sag=variables["lmax_sag"],
        pv_nivel=variables["nivel"],     lmin_nivel=variables["lmin_nivel"], lmax_nivel=variables["lmax_nivel"],
        pv_p80=variables["p80"],         lmax_p80=variables["lmax_p80"],
        pv_pres=variables["presion"],    lmax_pres=variables["lmax_pres"],
        pv_dens=variables["densidad"],   lmax_dens=variables["lmax_dens"]
    )

    salida_motor = motor_reglas_test(result, REGLAS_TEST, tiempo_actual)
    accion_real = defuzzy(result, salida_motor)

    return {
        "fuzzy": result,
        "regla": salida_motor,
        "accion_fisica": accion_real
    }



#Simulacion

In [30]:
# ============================================================
# DF DE PRUEBA MANUAL — 4 FILAS, 4 REGLAS
# ============================================================
# ==========================
# DF DE PRUEBA — 4 CASOS
# ==========================
df_prueba = pd.DataFrame([
    # Fila 0 → Regla 1: pot_sag ALTO y p80 ALTO
    {
        "pot_bolas": 740, "lmin_bolas": 700,
        "pot_sag": 30,   "lmax_sag": 24,    # offset = -6 → ALTO dominante
        "nivel": 60,     "lmin_nivel": 50,  "lmax_nivel": 70,
        "p80": 200,      "lmax_p80": 300,   # offset = 100 → ALTO dominante
        "presion": 15,   "lmax_pres": 25,   # offset = 10 → OK
        "densidad": 1.35, "lmax_dens": 1.60,
    },
    # Fila 1 → Regla 2: pot_sag OK y p80 OK
    {
        "pot_bolas": 740, "lmin_bolas": 700,
        "pot_sag": 22,   "lmax_sag": 24,    # offset ≈ 2 → OK dominante
        "nivel": 60,     "lmin_nivel": 50,  "lmax_nivel": 70,
        "p80": 280,      "lmax_p80": 300,   # offset = 20 → OK dominante
        "presion": 15,   "lmax_pres": 25,   # OK
        "densidad": 1.35, "lmax_dens": 1.60,
    },
    # Fila 2 → Regla 3: p80 BAJO y presion OK
    {
        "pot_bolas": 740, "lmin_bolas": 700,
        "pot_sag": 22,   "lmax_sag": 24,
        "nivel": 60,     "lmin_nivel": 50,  "lmax_nivel": 70,
        "p80": 320,      "lmax_p80": 300,   # offset = -20 → BAJO dominante
        "presion": 15,   "lmax_pres": 25,   # OK
        "densidad": 1.35, "lmax_dens": 1.60,
    },
    # Fila 3 → Regla 4: p80 ALTO y presion ALTA
    {
        "pot_bolas": 740, "lmin_bolas": 700,
        "pot_sag": 22,   "lmax_sag": 24,
        "nivel": 60,     "lmin_nivel": 50,  "lmax_nivel": 70,
        "p80": 200,      "lmax_p80": 300,   # ALTO
        "presion": 5,    "lmax_pres": 25,   # offset = 20 → ALTO dominante
        "densidad": 1.35, "lmax_dens": 1.60,
    },
])


# Resetear cooldowns antes de probar
cooldown_until_test.clear()

historial_prueba = []
tiempo_simulado = 0.0
SCAN = 5.0

for i in range(len(df_prueba)):
    vars_proceso = df_prueba.iloc[i].to_dict()
    out = tick_test(vars_proceso, tiempo_simulado)
    acc = out.get("accion_fisica", {})

    historial_prueba.append({
        "i": i,
        "tiempo_s": tiempo_simulado,
        "regla": out["regla"]["regla_activada"],
        "bloque": out["regla"]["bloque"],
        "accion": acc.get("accion", None),
        "step": acc.get("step", 0),
        "fuerza": acc.get("fuerza", 0),
    })

    tiempo_simulado += SCAN

df_hist_prueba = pd.DataFrame(historial_prueba)
print(df_hist_prueba)


   i  tiempo_s  regla             bloque                   accion  step  \
0  0       0.0      1  potencia_sag_test        aumentar_tonelaje   8.0   
1  1       5.0      2  potencia_sag_test       disminuir_tonelaje  -6.0   
2  2      10.0      3   control_p80_test   aumentar_agua_ciclones   3.0   
3  3      15.0      4   control_p80_test  disminuir_agua_ciclones  -6.0   

   fuerza  
0     0.8  
1     0.6  
2     0.5  
3     1.0  
